# Supplementary — Loss Taxonomy

Explain where agreements drop between successive rungs (2→3, 3→4, 4→5), partitioning losses into interpretable categories.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

OUTPUT_DIR  = Path('../results')
QC_DIR      = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUMMARY_DIR = OUTPUT_DIR / 'summary_stats'
FIGURE_DIR  = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(exist_ok=True, parents=True)

# Parameters
RBH_COVERAGE_THRESHOLD = float(globals().get('RBH_COVERAGE_THRESHOLD', 0.95))
RUNG4_DENOM = globals().get('RUNG4_DENOM', 'monotonic')
SAMPLE_RBH = int(globals().get('SAMPLE_RBH', 0))  # 0 = no sampling


In [ ]:
def _bool(x):
    return {True: True, False: False, 'True': True, 'False': False}.get(x, False)

from collections import defaultdict

counts = defaultdict(int)
assemblies = set()

rbh_files = sorted(RESULTS_DIR.rglob('*.gene_pairs_rbh.tsv'))

for rbh_path in rbh_files:
    # Accession detection from filename prefix
    acc = rbh_path.name.split('.')[0]
    assemblies.add(acc)

    try:
        rbh = pd.read_csv(rbh_path, sep='	')
    except Exception as e:
        warnings.warn(f'RBH read failed {rbh_path}: {e}')
        continue

    if not set(['frac_ensembl_covered','frac_cat_covered']).issubset(rbh.columns):
        continue

    r2 = rbh[(rbh['frac_ensembl_covered'] >= RBH_COVERAGE_THRESHOLD) & (rbh['frac_cat_covered'] >= RBH_COVERAGE_THRESHOLD)]
    r2_genes = set(r2['ensembl_gene_id'])
    counts['r2_denom'] += len(r2_genes)

    # 2→3: transcript concordance failure reasons
    tc_path = None
    # Find by matching accession prefix in filename set
    for p in QC_DIR.rglob(f"{acc}_transcript_concordance.tsv"):
        tc_path = p
        break
    if tc_path is None:
        continue
    tc = pd.read_csv(tc_path, sep='	')
    if 'ensembl_gene_id' not in tc.columns:
        continue
    tc = tc[tc['ensembl_gene_id'].isin(r2_genes)]

    # Define categories
    exact = (tc['n_ens_exact'] >= 1) | (tc.get('n_cat_exact', 0) >= 1)
    n_tx0 = (tc.get('n_ensembl_transcripts', 1) == 0) | (tc.get('n_cat_transcripts', 1) == 0)
    partial_sum = 0
    for col in ['n_ens_intron_match','n_ens_subset','n_ens_superset','n_ens_partial_5','n_ens_partial_3','n_ens_other_partial',
                'n_cat_intron_match','n_cat_subset','n_cat_superset','n_cat_partial_5','n_cat_partial_3','n_cat_other_partial']:
        if col in tc.columns:
            partial_sum = partial_sum + tc[col].fillna(0)
    partial_or_intron = (partial_sum > 0) & (~exact)
    other_no_match = (~exact) & (~partial_or_intron) & (~n_tx0)

    counts['r3_pass'] += int(exact.sum())
    counts['r2to3_no_tx'] += int(n_tx0.sum())
    counts['r2to3_partial'] += int(partial_or_intron.sum())
    counts['r2to3_nomatch'] += int(other_no_match.sum())

    # 3→4 using coding integrity
    r3_genes = set(tc.loc[exact, 'ensembl_gene_id'])
    ci_path = None
    for p in QC_DIR.rglob(f"{acc}_coding_integrity.tsv"):
        ci_path = p
        break
    if ci_path is None:
        continue
    ci = pd.read_csv(ci_path, sep='	')
    if 'ensembl_gene_id' not in ci.columns:
        continue
    for col in ['has_ensembl_cds','has_cat_cds','start_codon_match','stop_codon_match','frameshift_detected']:
        if col in ci.columns:
            ci[col] = ci[col].map({'True': True, 'False': False, True: True, False: False})
    ci = ci[ci['ensembl_gene_id'].isin(r3_genes)]

    has_e = ci['has_ensembl_cds'] == True
    has_c = ci['has_cat_cds'] == True
    both  = has_e & has_c

    counts['r3_denom'] += len(r3_genes)
    counts['r3to4_no_cds_ens']  += int((~has_e & has_c).sum())
    counts['r3to4_no_cds_cat']  += int((has_e & ~has_c).sum())
    counts['r3to4_no_cds_both'] += int((~has_e & ~has_c).sum())

    start_ok = ci['start_codon_match'] == True
    stop_ok  = ci['stop_codon_match'] == True
    pass4    = both & start_ok & stop_ok
    counts['r4_pass'] += int(pass4.sum())
    counts['r3to4_boundary_mismatch'] += int((both & (~(start_ok & stop_ok))).sum())

    # 4→5 frameshift
    r4_genes = set(ci.loc[pass4, 'ensembl_gene_id'])
    ci_r4    = ci[ci['ensembl_gene_id'].isin(r4_genes)]
    fs = ci_r4['frameshift_detected'] == True
    counts['r4_denom'] += len(ci_r4)
    counts['r5_pass'] += int((~fs).sum())
    counts['r4to5_frameshift'] += int(fs.sum())

n_asm = len(assemblies)
print(f'Assemblies considered: {n_asm}')

# Build stacked waterfall percentages for each transition
transitions = []
# 2→3
if counts['r2_denom'] > 0:
    denom = counts['r2_denom']
    transitions.append(('2→3',
                        counts['r3_pass']/denom*100,
                        {
                          'Partial/Intron-level': counts['r2to3_partial']/denom*100,
                          'No transcripts parsed': counts['r2to3_no_tx']/denom*100,
                          'No match': counts['r2to3_nomatch']/denom*100,
                        }))
# 3→4
if counts['r3_denom'] > 0:
    denom = counts['r3_denom']
    transitions.append(('3→4',
                        counts['r4_pass']/denom*100,
                        {
                          'No CDS (Ensembl only)': counts['r3to4_no_cds_ens']/denom*100,
                          'No CDS (CAT only)': counts['r3to4_no_cds_cat']/denom*100,
                          'No CDS (both)': counts['r3to4_no_cds_both']/denom*100,
                          'Start/stop mismatch': counts['r3to4_boundary_mismatch']/denom*100,
                        }))
# 4→5
if counts['r4_denom'] > 0:
    denom = counts['r4_denom']
    transitions.append(('4→5',
                        counts['r5_pass']/denom*100,
                        {'Frameshift present': counts['r4to5_frameshift']/denom*100}))

# Plot
fig, ax = plt.subplots(figsize=(7.2, 3.2))
y = np.arange(len(transitions))
bar_h = 0.55
palette = {
    'Partial/Intron-level':'#8da0cb',
    'No transcripts parsed':'#a6d854',
    'No match':'#c7c7c7',
    'No CDS (Ensembl only)':'#66c2a5',
    'No CDS (CAT only)':'#fc8d62',
    'No CDS (both)':'#bdbdbd',
    'Start/stop mismatch':'#e78ac3',
    'Frameshift present':'#e5c494',
}

for i, (label, pass_pct, losses) in enumerate(transitions):
    # Pass segment (left)
    ax.barh(i, pass_pct, height=bar_h, color='#4daf4a', alpha=0.85, label='Pass' if i==0 else None)
    left = pass_pct
    for name, pct in losses.items():
        ax.barh(i, pct, left=left, height=bar_h, color=palette.get(name,'#cccccc'), label=name if i==0 else None, alpha=0.85)
        left += pct
    ax.text(pass_pct - 1.0, i, f'{pass_pct:.1f}%', va='center', ha='right', fontsize=7, color='white', fontweight='bold')

ax.set_yticks(y)
ax.set_yticklabels([t[0] for t in transitions])
ax.set_xlim(0, 100)
ax.set_xlabel('Share of previous rung (%)')
ax.set_title('Where agreement drops between rungs (all assemblies)')
ax.spines[['top','right']].set_visible(False)
ax.legend(loc='lower right', fontsize=7, ncol=2, frameon=False)

# Save
_tag = f"rbh{int(RBH_COVERAGE_THRESHOLD*100)}"
fig.savefig(FIGURE_DIR / f"figure_loss_taxonomy_{_tag}.svg", bbox_inches='tight')
fig.savefig(FIGURE_DIR / f"figure_loss_taxonomy_{_tag}.png", dpi=300, bbox_inches='tight')
print('Saved loss taxonomy figure')
